# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [ ]:
from pathlib import Path
import gc
import math
import shutil

import duckdb
import polars as pl
import pyarrow.parquet as pq

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(60)

# --- Path -------------------------------------------------------------
def cari_root(mulai):
    """Naik dari folder saat ini sampai menemukan akar repo (berisi requirements.txt dan Dockerfile).
    Lebih andal daripada memeriksa folder `data`, yang bisa ikut ada di dalam notebooks/."""
    for p in [mulai, *mulai.parents]:
        if (p / 'requirements.txt').exists() and (p / 'Dockerfile').exists():
            return p
    return mulai


PROJECT_ROOT = cari_root(Path.cwd().resolve())

DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'ID_REG_KG_2511'
DATA_PATH = str(DATA_DIR / '*.parquet')

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'dataset_bersih'            # hasil utama (per file)
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset_bersih.parquet'   # opsional (1 file)

def parquet_valid(path):
    """True jika footer Parquet terbaca (berkas rusak/terpotong akibat unduhan gagal dilewati)."""
    try:
        pq.ParquetFile(path)
        return True
    except Exception:
        return False


semua_file = sorted(DATA_DIR.glob('*.parquet'))
valid_files = [f for f in semua_file if f.stat().st_size > 0 and parquet_valid(f)]
file_dilewati = [f.name for f in semua_file if f not in valid_files]
if file_dilewati:
    print('PERINGATAN - berkas dilewati (rusak/kosong):', file_dilewati)
if not valid_files:
    raise FileNotFoundError(
        f'Dataset tidak ditemukan di {DATA_DIR}. '
        'Pastikan Milestone 1 sudah menempatkan data di data/raw/.'
    )

# --- LazyFrame: hanya membaca metadata, data TIDAK dimuat ke RAM ---------
df = pl.scan_parquet([str(f) for f in valid_files])
SCHEMA = df.collect_schema()
SEMUA_KOLOM = SCHEMA.names()

# --- Pengelompokan kolom (dipakai di seluruh notebook) -----------------
KOLOM_VEKTOR = [c for c in ('embedding', 'tfidf_vector') if c in SEMUA_KOLOM]
KOLOM_JSON = [c for c in SEMUA_KOLOM if c.endswith('_json')]
KOLOM_TEKS_PANJANG = [c for c in ('about', 'content') if c in SEMUA_KOLOM]
KOLOM_KATEGORIK = [c for c in (
    'regulation_type', 'enacting_body', 'regulation_number', 'year',
    'effective_date', 'chapter', 'article', 'kg_primary_domain') if c in SEMUA_KOLOM]
KOLOM_SKOR = [c for c in (
    'kg_domain_confidence', 'kg_authority_score', 'kg_temporal_score',
    'kg_legal_richness', 'kg_legal_complexity', 'kg_completeness_score',
    'kg_connectivity_score') if c in SEMUA_KOLOM]               # seharusnya 0..1
KOLOM_HITUNG = [c for c in ('kg_entity_count', 'kg_cross_ref_count') if c in SEMUA_KOLOM]

# Nilai yang sebenarnya berarti "kosong" tetapi bukan NULL
PLACEHOLDER = ['', 'unknown', 'n/a', 'null', 'none', '-', 'tidak diketahui']

print('PROJECT_ROOT :', PROJECT_ROOT)
print('DATA_DIR     :', DATA_DIR)
print('Jumlah file  :', len(valid_files))
print('Jumlah kolom :', len(SEMUA_KOLOM))
print('Vektor       :', KOLOM_VEKTOR)
print('JSON         :', len(KOLOM_JSON), 'kolom')
print('Skor (0..1)  :', KOLOM_SKOR)
print('Kolom hitung :', KOLOM_HITUNG)


def jalankan_per_file(ekspresi, transform=None):
    """Hitung `ekspresi` (agregat 1 baris) pada tiap file satu per satu, lalu jumlahkan.
    `transform` (opsional) = fungsi cleaning yang diterapkan ke tiap file sebelum dihitung.
    Dengan cara ini RAM hanya terpakai sebesar 1 file."""
    hasil = []
    for file in valid_files:
        lf = pl.scan_parquet(file)
        if transform is not None:
            lf = transform(lf)
        hasil.append(lf.select(ekspresi).collect())
    gc.collect()
    return pl.concat(hasil).sum()

## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [ ]:
con = duckdb.connect()
con.execute("SET memory_limit = '2GB'")
con.execute("SET threads = 2")
con.execute("SET preserve_insertion_order = false")
con.execute("SET temp_directory = '/tmp/duckdb_temp'")

kolom_ringan = [c for c in SEMUA_KOLOM if c not in KOLOM_VEKTOR + KOLOM_JSON + KOLOM_TEKS_PANJANG]
select_ringan = ', '.join(f'"{c}"' for c in kolom_ringan)
glob_sql = DATA_PATH.replace("'", "''")

parquet_files = [str(p) for p in valid_files]

if not parquet_files:
    raise FileNotFoundError(f"Tidak ada file Parquet valid di {DATA_DIR}")

# Buat string list path untuk query DuckDB dengan aman
read_list = ", ".join(f"'{p}'" for p in parquet_files)

profil = con.execute(
    f"""
    SUMMARIZE
    SELECT {select_ringan},
           length(CAST(about AS VARCHAR)) AS about_len,
           length(CAST(content AS VARCHAR)) AS content_len
    FROM read_parquet([{read_list}])
    """
).pl()

# Tampilkan versi ringkas (nilai min/max teks dipotong supaya tabel terbaca)
profil_ringkas = profil.select(
    'column_name', 'column_type',
    pl.col('min').str.slice(0, 25),
    pl.col('max').str.slice(0, 25),
    'approx_unique', 'avg', 'std', 'q50', 'count',
    pl.col('null_percentage').cast(pl.Float64),
)
profil_ringkas

## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [ ]:
KOLOM_PERIKSA_PLACEHOLDER = KOLOM_KATEGORIK + KOLOM_TEKS_PANJANG


def adalah_placeholder(kolom):
    return pl.col(kolom).str.strip_chars().str.to_lowercase().is_in(PLACEHOLDER)


# Kolom vektor (embedding, tfidf_vector) sangat besar: dihitung satu kolom per kali agar RAM aman.
KOLOM_NON_VEKTOR = [c for c in SEMUA_KOLOM if c not in KOLOM_VEKTOR]

ekspresi = [pl.len().alias('__baris__')]
ekspresi += [pl.col(c).null_count().alias(f'null__{c}') for c in KOLOM_NON_VEKTOR]
ekspresi += [adalah_placeholder(c).sum().alias(f'ph__{c}') for c in KOLOM_PERIKSA_PLACEHOLDER]

hasil = jalankan_per_file(ekspresi).row(0, named=True)
for c in KOLOM_VEKTOR:
    hasil.update(jalankan_per_file([
        pl.col(c).null_count().alias(f'null__{c}'),
        (pl.col(c).list.len() == 0).sum().alias(f'kosong__{c}'),
    ]).row(0, named=True))
jumlah_baris = int(hasil['__baris__'])
print('Jumlah seluruh baris:', f'{jumlah_baris:,}')

skema_ringkasan = {'kolom': pl.String, 'jumlah': pl.Int64, 'persen': pl.Float64}


def buat_ringkasan(prefiks, daftar_kolom, nama_jumlah, nama_persen):
    baris = [
        {'kolom': c, 'jumlah': int(hasil[f'{prefiks}{c}']),
         'persen': round(int(hasil[f'{prefiks}{c}']) / jumlah_baris * 100, 2)}
        for c in daftar_kolom if int(hasil[f'{prefiks}{c}']) > 0
    ]
    return (pl.DataFrame(baris, schema=skema_ringkasan)
            .rename({'jumlah': nama_jumlah, 'persen': nama_persen})
            .sort(nama_jumlah, descending=True))


# (1) NULL asli
null_summary = buat_ringkasan('null__', SEMUA_KOLOM, 'jumlah_null', 'persen_null')
print('Kolom dengan NULL asli:', null_summary.height)
display(null_summary)

# (2) Missing tersembunyi ('Unknown', '', spasi, dst.)
placeholder_summary = buat_ringkasan('ph__', KOLOM_PERIKSA_PLACEHOLDER, 'jumlah_placeholder', 'persen_placeholder')
print('Kolom dengan missing tersembunyi:', placeholder_summary.height)
display(placeholder_summary)

# (3) List kosong pada kolom vektor
for c in KOLOM_VEKTOR:
    print(f'{c}: list kosong = {int(hasil[f"kosong__{c}"]):,} | NULL = {int(hasil[f"null__{c}"]):,}')

# Isi kolom yang bermasalah (untuk melihat bentuk missing tersembunyinya)
if 'regulation_type' in SEMUA_KOLOM:
    display(df.group_by('regulation_type').len().sort('len', descending=True).collect())

## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [ ]:
KOLOM_PLACEHOLDER = placeholder_summary['kolom'].to_list()

# Median untuk kolom numerik yang punya NULL asli (kosong jika tidak ada)
KOLOM_NULL_NUMERIK = [c for c in null_summary['kolom'].to_list()
                      if c not in KOLOM_VEKTOR + KOLOM_JSON and SCHEMA[c].is_numeric()]
MEDIAN_NUMERIK = {c: df.select(pl.col(c).median()).collect().item() for c in KOLOM_NULL_NUMERIK}


def tangani_missing(lf):
    ekspresi = [
        pl.when(adalah_placeholder(c)).then(None).otherwise(pl.col(c)).alias(c)
        for c in KOLOM_PLACEHOLDER
    ]
    ekspresi += [pl.col(c).fill_null(m) for c, m in MEDIAN_NUMERIK.items()]
    return lf.with_columns(ekspresi) if ekspresi else lf


df_missing_handled = tangani_missing(df)

# Verifikasi: hitung NULL pada kolom yang ditangani, sebelum vs sesudah
if KOLOM_PLACEHOLDER:
    sesudah = jalankan_per_file(
        [pl.col(c).null_count().alias(c) for c in KOLOM_PLACEHOLDER],
        transform=tangani_missing,
    ).row(0, named=True)
    perbandingan = pl.DataFrame([
        {'kolom': c,
         'null_sebelum': int(hasil[f'null__{c}']),
         'placeholder_diubah': int(hasil[f'ph__{c}']),
         'null_sesudah': int(sesudah[c])}
        for c in KOLOM_PLACEHOLDER
    ])
    display(perbandingan)
else:
    print('Tidak ada missing tersembunyi yang perlu diubah.')

print('NULL numerik yang diisi median:', MEDIAN_NUMERIK or 'tidak ada')

## 3b. Penanganan Tahun Tidak Valid

Kolom `year` berisi nilai di luar rentang yang masuk akal (hasil `SUMMARIZE` menunjukkan nilai minimum `0000` dan maksimum `2030`). Strategi: nilai di luar rentang 1945 sampai 2026 diubah menjadi NULL (baris **tidak dibuang**), dan penanda `year_diperbaiki` disimpan agar bisa ditelusuri. Analisis temporal pada Milestone 3 hanya memakai baris dengan `year` valid.

In [ ]:
TAHUN_MIN, TAHUN_MAX = 1945, 2026


def tahun_tidak_valid():
    """True jika `year` terisi tetapi bukan angka tahun dalam rentang TAHUN_MIN..TAHUN_MAX."""
    tahun = pl.col('year').cast(pl.Int64, strict=False)
    return pl.col('year').is_not_null() & ~tahun.is_between(TAHUN_MIN, TAHUN_MAX).fill_null(False)


def tangani_year(lf):
    """Tahun tidak valid -> NULL; penandanya disimpan di kolom `year_diperbaiki`."""
    return lf.with_columns(
        tahun_tidak_valid().alias('year_diperbaiki')
    ).with_columns(
        pl.when(pl.col('year_diperbaiki')).then(None).otherwise(pl.col('year')).alias('year')
    )


sebaran_year_salah = df.filter(tahun_tidak_valid()).group_by('year').len().sort('len', descending=True).collect()
total_year_salah = int(sebaran_year_salah['len'].sum()) if sebaran_year_salah.height else 0
print(f'Baris dengan year tidak valid: {total_year_salah:,} '
      f'({total_year_salah / jumlah_baris * 100:.3f}% dari {jumlah_baris:,})')
display(sebaran_year_salah.head(20))

## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [ ]:
# --- A. Kolom hitungan: batas IQR (dihitung dari seluruh data, 2 kolom Int64 saja = ringan) ---
kuantil = df_missing_handled.select(
    [pl.col(c).quantile(0.25).alias(f'{c}__q1') for c in KOLOM_HITUNG]
    + [pl.col(c).quantile(0.75).alias(f'{c}__q3') for c in KOLOM_HITUNG]
    + [pl.col(c).quantile(0.99).alias(f'{c}__p99') for c in KOLOM_HITUNG]
    + [pl.col(c).max().alias(f'{c}__max') for c in KOLOM_HITUNG]
).collect().row(0, named=True)

BATAS_OUTLIER = {}      # {kolom: (batas_bawah, batas_atas)} sebagai bilangan bulat
for c in KOLOM_HITUNG:
    q1, q3 = kuantil[f'{c}__q1'], kuantil[f'{c}__q3']
    iqr = q3 - q1
    bawah = max(0, math.ceil(q1 - 1.5 * iqr))                   # hitungan tidak boleh negatif
    atas = math.floor(q3 + 1.5 * iqr) if iqr > 0 else int(kuantil[f'{c}__p99'])
    BATAS_OUTLIER[c] = (bawah, atas)
    print(f'{c}: Q1={q1}, Q3={q3}, IQR={iqr} -> batas bawah={bawah}, batas atas={atas}')

jumlah_outlier = df_missing_handled.select(
    [((pl.col(c) < b) | (pl.col(c) > a)).sum().alias(c) for c, (b, a) in BATAS_OUTLIER.items()]
).collect().row(0, named=True)

tabel_outlier = pl.DataFrame([
    {'kolom': c,
     'batas_bawah': b, 'batas_atas': a,
     'nilai_max': int(kuantil[f'{c}__max']),
     'jumlah_outlier': int(jumlah_outlier[c]),
     'persen_outlier': round(int(jumlah_outlier[c]) / jumlah_baris * 100, 2)}
    for c, (b, a) in BATAS_OUTLIER.items()
])
display(tabel_outlier)

# --- B. Kolom skor: periksa pelanggaran rentang 0..1 -------------------------------------
pelanggaran_skor = df_missing_handled.select(
    [((pl.col(c) < 0) | (pl.col(c) > 1)).sum().alias(c) for c in KOLOM_SKOR]
).collect().row(0, named=True)
nilai_max_skor = df_missing_handled.select([pl.col(c).max().alias(c) for c in KOLOM_SKOR]).collect().row(0, named=True)

tabel_skor = pl.DataFrame([
    {'kolom': c, 'nilai_max': nilai_max_skor[c], 'di_luar_0_1': int(pelanggaran_skor[c])}
    for c in KOLOM_SKOR
])
display(tabel_skor)


# --- C. Penanganan: windsorization (clip), bukan menghapus baris ---------------------------
def tangani_outlier(lf):
    penanda = [((pl.col(c) < b) | (pl.col(c) > a)).alias(f'{c}_outlier')       # dihitung SEBELUM clip
               for c, (b, a) in BATAS_OUTLIER.items()]
    clip_hitungan = [pl.col(c).clip(lower_bound=b, upper_bound=a)
                     for c, (b, a) in BATAS_OUTLIER.items()]
    clip_skor = [pl.col(c).clip(lower_bound=0.0, upper_bound=1.0) for c in KOLOM_SKOR]
    return lf.with_columns(penanda).with_columns(clip_hitungan + clip_skor)


df_outlier_handled = tangani_outlier(df_missing_handled)

## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [ ]:
KUNCI_DEDUP = [c for c in ('regulation_type', 'regulation_number', 'year',
                           'article', 'chunk_id', 'content') if c in SEMUA_KOLOM]


def ekspresi_hash(daftar_kolom):
    return pl.concat_str(
        [pl.col(c).cast(pl.String).fill_null('<NULL>') for c in daftar_kolom],
        separator='\x1f',
    ).hash(seed=0)


# Bangun tabel hash per file (RAM hanya sebesar 1 file; hasil akhirnya kecil)
potongan = []
for file in valid_files:
    potongan.append(
        tangani_missing(pl.scan_parquet(file)).select(
            'global_id',
            ekspresi_hash(KUNCI_DEDUP).alias('hash_kunci'),
            ekspresi_hash(['content']).alias('hash_content'),
        ).collect()
    )
tabel_hash = pl.concat(potongan)
del potongan
gc.collect()

total_baris = tabel_hash.height
unik_global_id = tabel_hash['global_id'].n_unique()
unik_kunci = tabel_hash['hash_kunci'].n_unique()
unik_content = tabel_hash['hash_content'].n_unique()

print(f'Total baris                              : {total_baris:,}')
print(f'Baris unik berdasarkan global_id         : {unik_global_id:,}  (ID surrogate, selalu unik)')
print(f'Baris unik berdasarkan kunci bisnis      : {unik_kunci:,}')
print(f'Duplikat berdasarkan kunci bisnis        : {total_baris - unik_kunci:,}')
print(f'(info) duplikat jika hanya content sama  : {total_baris - unik_content:,}')

# Pertahankan baris pertama (global_id terkecil) tiap kunci, buang sisanya
pertahankan = tabel_hash.sort('global_id').unique(subset='hash_kunci', keep='first', maintain_order=True)
ID_DIBUANG = (tabel_hash.join(pertahankan.select('global_id'), on='global_id', how='anti')
              .select('global_id'))
print(f'Baris yang akan dibuang                  : {ID_DIBUANG.height:,}')

if ID_DIBUANG.height > 0:   # tampilkan contoh baris duplikat yang dibuang
    display(
        df_missing_handled
        .join(ID_DIBUANG.head(5).lazy(), on='global_id', how='semi')
        .select('global_id', *[c for c in KUNCI_DEDUP if c != 'content'],
                pl.col('content').str.slice(0, 80).alias('content_awal'))
        .collect()
    )


def hapus_duplikat(lf):
    return lf.join(ID_DIBUANG.lazy(), on='global_id', how='anti')


def terapkan_cleaning(lf):
    """Seluruh aturan cleaning dalam satu fungsi (dipakai di validasi dan saat menyimpan)."""
    return hapus_duplikat(tangani_year(tangani_outlier(tangani_missing(lf))))


df_dedup = hapus_duplikat(df_outlier_handled)

## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [ ]:
# Tahun acuan untuk kg_years_old: ditentukan dari data (nilai terbanyak dari year + kg_years_old)
tahun_acuan = (
    df.select((pl.col('year').cast(pl.Int64, strict=False) + pl.col('kg_years_old')).alias('tahun_acuan'))
    .drop_nulls().group_by('tahun_acuan').len().sort('len', descending=True).head(3).collect()
)
display(tahun_acuan)
REF_TAHUN = int(tahun_acuan['tahun_acuan'][0])
print('Tahun acuan kg_years_old:', REF_TAHUN)


def ekspresi_validasi():
    tahun = pl.col('year').cast(pl.Int64, strict=False)
    return [
        pl.len().alias('total_baris'),
        pl.sum_horizontal(
            [adalah_placeholder(c).fill_null(False).cast(pl.Int64) for c in KOLOM_PERIKSA_PLACEHOLDER]
        ).sum().alias('teks_placeholder'),
        pl.sum_horizontal(
            [((pl.col(c) < 0) | (pl.col(c) > 1)).fill_null(False).cast(pl.Int64) for c in KOLOM_SKOR]
        ).sum().alias('skor_di_luar_0_1'),
        tahun_tidak_valid().sum().alias('year_tidak_valid'),
        ((tahun + pl.col('kg_years_old')) != REF_TAHUN).fill_null(False).sum().alias('years_old_tidak_konsisten'),
        (pl.col('content').str.strip_chars().str.len_chars() == 0).fill_null(True).sum().alias('content_kosong_atau_null'),
        pl.sum_horizontal(
            [(pl.col(c) < 0).fill_null(False).cast(pl.Int64) for c in KOLOM_HITUNG]
        ).sum().alias('hitungan_negatif'),
        pl.sum_horizontal(
            [((pl.col(c) < b) | (pl.col(c) > a)).fill_null(False).cast(pl.Int64)
             for c, (b, a) in BATAS_OUTLIER.items()]
        ).sum().alias('hitungan_di_atas_batas_iqr'),
    ]


sebelum = jalankan_per_file(ekspresi_validasi()).row(0, named=True)
sesudah = jalankan_per_file(ekspresi_validasi(), transform=terapkan_cleaning).row(0, named=True)

# Aturan duplikat dihitung dari tabel hash (lintas file)
tabel_sisa = tabel_hash.join(ID_DIBUANG, on='global_id', how='anti')
sebelum['duplikat_kunci_bisnis'] = total_baris - unik_kunci
sesudah['duplikat_kunci_bisnis'] = tabel_sisa.height - tabel_sisa['hash_kunci'].n_unique()

validasi = pl.DataFrame([
    {'aturan': k,
     'sebelum': int(sebelum[k]),
     'sesudah': int(sesudah[k]),
     'status': '-' if k == 'total_baris' else ('OK' if int(sesudah[k]) == 0 else 'PERIKSA')}
    for k in sebelum
])
validasi

## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

# Bebaskan memori dari langkah sebelumnya (tidak dipakai lagi di langkah ini)
for nama in ('con', 'tabel_sisa', 'tabel_hash', 'pertahankan', 'sebelum', 'sesudah'):
    globals().pop(nama, None)
gc.collect()

ULANG_DARI_AWAL = False     # True = hapus hasil lama dan proses ulang semua file (pakai jika aturan cleaning berubah)
UKURAN_BATCH = 1000         # baris per batch; turunkan (mis. 500) jika memori masih habis

if ULANG_DARI_AWAL and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for i, file in enumerate(valid_files, start=1):
    tujuan = OUTPUT_DIR / file.name
    if tujuan.exists():                      # file yang sudah selesai dilewati (bisa dilanjutkan setelah crash)
        continue
    sementara = tujuan.with_suffix('.tmp')   # tulis ke .tmp dulu, ganti nama setelah selesai

    penulis = None
    for batch in pq.ParquetFile(file).iter_batches(batch_size=UKURAN_BATCH):
        bersih = terapkan_cleaning(pl.from_arrow(batch).lazy()).collect()
        tabel = bersih.to_arrow()
        if penulis is None:
            penulis = pq.ParquetWriter(sementara, tabel.schema, compression='zstd')
        penulis.write_table(tabel)
        del batch, bersih, tabel
    if penulis is not None:
        penulis.close()
        sementara.rename(tujuan)
    gc.collect()

    if i % 5 == 0 or i == len(valid_files):
        print(f'Berhasil: {i}/{len(valid_files)}')

# Verifikasi hasil simpan
jumlah_baris_akhir = (pl.scan_parquet(str(OUTPUT_DIR / '*.parquet'))
                      .select(pl.len()).collect().item())
jumlah_dibuang = jumlah_baris - jumlah_baris_akhir
ukuran_gb = sum(f.stat().st_size for f in OUTPUT_DIR.glob('*.parquet')) / 1e9

print(f'Baris sebelum cleaning : {jumlah_baris:,}')
print(f'Baris sesudah cleaning : {jumlah_baris_akhir:,}')
print(f'Baris dibuang          : {jumlah_dibuang:,}')
print(f'Ukuran folder          : {ukuran_gb:.2f} GB')
print(f'Data bersih tersimpan di: {OUTPUT_DIR}')

assert jumlah_dibuang == ID_DIBUANG.height, 'Jumlah baris dibuang tidak sesuai dengan hasil deduplikasi'
print('Kolom baru:', [c for c in pl.scan_parquet(str(OUTPUT_DIR / '*.parquet')).collect_schema().names()
                      if c not in SEMUA_KOLOM])

## Catatan Milestone 2

> Angka di bawah berasal dari eksekusi pada **1 berkas Parquet (10.000 baris)**. Jalankan ulang seluruh notebook pada dataset lengkap, lalu perbarui angka-angkanya.

**Dimensi kualitas yang paling bermasalah:** *missing tersembunyi*. Tidak ada NULL asli, tetapi 6.071 sel berisi placeholder (`Unknown`, kosong, `-`, dst.) pada 5 kolom: `effective_date` 2.810 (28,10%), `chapter` 1.788 (17,88%), `regulation_type` 806 (8,06%), `enacting_body` 460 (4,60%), dan `regulation_number` 207 (2,07%).

**Strategi missing values:** placeholder diubah menjadi NULL agar bisa dihitung sebagai missing. Tidak ada imputasi karena kolom terdampak bersifat kategorik/identitas dan tidak ada kolom numerik yang NULL.

**Strategi outlier:** batas IQR (1,5 x IQR) untuk kolom hitungan, ditangani dengan *winsorization* (`clip`), bukan menghapus baris. `kg_entity_count` dibatasi 0 sampai 39 (995 baris, 9,95%) dan `kg_cross_ref_count` 0 sampai 7 (1.002 baris, 10,02%); nilai asli ditandai pada kolom `*_outlier`. Pada kolom skor yang seharusnya 0 sampai 1, `kg_connectivity_score` melanggar pada 34 baris (maksimum 1,139) dan di-*clip* ke 1.

**Duplikat:** dihitung dengan kunci bisnis (`regulation_type`, `regulation_number`, `year`, `article`, `chunk_id`, `content`), bukan seluruh baris. 11 duplikat dibuang (baris dengan `global_id` terkecil dipertahankan). Sebanyak 881 baris memiliki `content` sama tetapi kunci bisnis berbeda, sehingga dipertahankan.

**Jumlah baris berubah:** 10.000 menjadi 9.989 (11 baris dibuang). Hasil disimpan per berkas di `data/processed/dataset_bersih/` (Parquet, kompresi zstd).

**Aturan validasi:** 8 aturan (placeholder teks, skor di luar 0..1, `year` 1945..2026, konsistensi `kg_years_old` terhadap tahun acuan 2024 yang diturunkan dari data, `content` kosong, hitungan negatif, hitungan di atas batas IQR, duplikat kunci bisnis) semuanya 0 pelanggaran setelah cleaning.

**Keterbatasan yang diketahui:** `regulation_type` masih memuat variasi penulisan `UU` (10 baris) di samping `UNDANG-UNDANG` (678 baris) yang belum dinormalisasi.